# Qwen2.5-1.5B-Instruct Inference
Loading and running the Qwen/Qwen2.5-1.5B-Instruct model using 🤗 Transformers.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer,TrainingArguments
import torch
from peft import Lo
model_id = 'Qwen/Qwen2.5-1.5B-Instruct'

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
print('Tokenizer loaded ✓')
print('Chat template:', tokenizer.chat_template)

In [ ]:
# Load model — uses GPU if available, otherwise CPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)
print('Model loaded ✓')
print('Device map:', model.hf_device_map)

In [ ]:
rank_dim=8
lora_alpha=5
lora_dropout=0.05

peft_config=LoraConfig()

In [ ]:
# Run a simple chat-style inference
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Explain quantum computing in one sentence."}
]

# Apply the chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([text], return_tensors='pt').to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

# Decode only the newly generated tokens
generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print('Response:', response)